In [3]:
import pandas as pd
import numpy as np

# Chargement du profil de vitesse WLTC (1 seule colonne, sans en-tete, en m/s)
wltc_raw = pd.read_excel('../data/wltc.xlsx', sheet_name='Feuil1', header=None)
wltc_raw.columns = ['speed']

print('Un seul cycle :', wltc_raw.shape, '| duree :', len(wltc_raw), 's')

# Repetition du cycle 7 fois (concatenation verticale)
df = pd.concat([wltc_raw] * 7, ignore_index=True)

# Colonne time : 0, 1, 2, ... N-1 (1 Hz constant)
df['time'] = np.arange(len(df), dtype=float)

print('Cycle repete 7 fois :', df.shape, '| duree totale :', len(df), 's =', round(len(df)/3600, 2), 'h')
print()
print(df.head(3))
print('...')
print(df.tail(3))

Un seul cycle : (1801, 1) | duree : 1801 s
Cycle repete 7 fois : (12607, 2) | duree totale : 12607 s = 3.5 h

   speed  time
0    0.0   0.0
1    0.0   1.0
2    0.0   2.0
...
       speed     time
12604    0.0  12604.0
12605    0.0  12605.0
12606    0.0  12606.0


In [ ]:
# Sauvegarde du cycle WLTC repete 7 fois (avant calcul des forces)
#output_file =  '../data/wltc_repeated_7x.csv'
#df.to_csv(output_file, index=False)

#print('Sauvegarde :', output_file)
print('Shape :', df.shape)

Sauvegarde : ../data/wltc_repeated_7x.csv
Shape : (12607, 2)


In [6]:
import numpy as np

m      = 1400.0
g      = 9.81
S      = 2.75
Cx     = 0.30
C0     = 0.008
C1     = 1.6e-6
alpha  = 0.0
rho    = 1.225

v = df['speed'].to_numpy(dtype=float)
t = df['time'].to_numpy(dtype=float)

dv, dt = np.diff(v), np.diff(t)
acc = np.divide(dv, dt, out=np.zeros_like(dv), where=dt != 0)
acc = np.concatenate(([0.0], acc))

df['hasAcceleration']      = acc.round(2)
df['hasAeroForce']         = (0.5 * rho * S * Cx * v**2).round(2)
df['hasRollingForce']      = (m * g * (C0 + C1 * v**2)).round(2)
df['hasGravityForce']      = round(float(m * g * np.sin(alpha)), 2)
df['hasAccelerationForce'] = (m * acc).round(2)
df['hasTotalForce']        = (df['hasAeroForce'] + df['hasRollingForce']
                              + df['hasGravityForce'] + df['hasAccelerationForce']).round(2)
df['hasPower'] = (df['hasTotalForce'] * df['speed']).round(2)

# verification energie globale (sur les 7 repetitions)
dtf = df['time'].diff().fillna(1).to_numpy()
print("Roulement  :", round(np.sum(df['hasRollingForce'].to_numpy()*v*dtf)/3.6e6, 2), "kWh (attendu ~4.7)")
print("Aero       :", round(np.sum(df['hasAeroForce'].to_numpy()*v*dtf)/3.6e6, 2), "kWh (~7)")
print("P_load     :", round(np.sum(df['hasPower'].to_numpy()*dtf)/3.6e6, 2), "kWh (~11-12)")

# verification des 6 points de jonction entre repetitions (indices 1801, 3602, 5403, 7204, 9005, 10806)
print()
print("Verification des jonctions entre repetitions :")
junction_indices = [1801 * k for k in range(1, 7)]
for idx in junction_indices:
    v_before = df['speed'].iloc[idx - 1]
    v_after = df['speed'].iloc[idx]
    acc_at_junction = df['hasAcceleration'].iloc[idx]
    print(f"  index {idx:5d} | v_avant={v_before:.4f} | v_apres={v_after:.4f} | acceleration calculee={acc_at_junction:.4f}")

Roulement  : 5.48 kWh (attendu ~4.7)
Aero       : 11.72 kWh (~7)
P_load     : 17.89 kWh (~11-12)

Verification des jonctions entre repetitions :
  index  1801 | v_avant=0.0000 | v_apres=0.0000 | acceleration calculee=0.0000
  index  3602 | v_avant=0.0000 | v_apres=0.0000 | acceleration calculee=0.0000
  index  5403 | v_avant=0.0000 | v_apres=0.0000 | acceleration calculee=0.0000
  index  7204 | v_avant=0.0000 | v_apres=0.0000 | acceleration calculee=0.0000
  index  9005 | v_avant=0.0000 | v_apres=0.0000 | acceleration calculee=0.0000
  index 10806 | v_avant=0.0000 | v_apres=0.0000 | acceleration calculee=0.0000


In [8]:
# Etape A : detection du decalage de distribution (WLTC vs train Artemis)
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')  # ajuste si ton chemin reel differe
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')  # ajuste le nom de fichier si besoin

def distribution_report(wltc_series, train_series, name):
    return {
        'variable': name,
        'wltc_min': float(wltc_series.min()), 'train_min': float(train_series.min()),
        'wltc_max': float(wltc_series.max()), 'train_max': float(train_series.max()),
        'wltc_mean': float(wltc_series.mean()), 'train_mean': float(train_series.mean()),
        'wltc_std': float(wltc_series.std()), 'train_std': float(train_series.std()),
        'pct_hors_domaine_train': float(
            100 * (~wltc_series.between(train_series.min(), train_series.max())).mean()
        ),
    }

shift_report = pd.DataFrame([
    distribution_report(df['hasPower'], train_df['hasPower'], 'hasPower'),
    distribution_report(df['speed'], train_df['speed'], 'speed'),
    distribution_report(df['hasAcceleration'], train_df['hasAcceleration'], 'hasAcceleration'),
])

display(shift_report)
print()
print(f"Resume : {shift_report['pct_hors_domaine_train'].max():.1f}% (au pire) des observations WLTC "
      f"sont hors du domaine observe pendant l'entrainement sur au moins une variable.")

,variable,wltc_min,train_min,wltc_max,train_max,wltc_mean,train_mean,wltc_std,train_std,pct_hors_domaine_train
0,hasPower,-26574.43,-46000.00,42190.280000,44230.330000,5108.906002,2922.156471,10816.632246,10240.715906,0.000000
1,speed,0.00,0.00,36.472222,30.972222,12.916374,9.854248,10.007906,7.954654,7.218212
2,hasAcceleration,-1.50,-4.08,1.750000,2.860000,0.000028,0.001703,0.531375,0.716133,0.000000



Resume : 7.2% (au pire) des observations WLTC sont hors du domaine observe pendant l'entrainement sur au moins une variable.


In [10]:
TABLES_DIR = Path('../results/tables')
shift_report.to_csv(TABLES_DIR / 'wltc_distribution_shift.csv', index=False)
print('Sauvegarde :', TABLES_DIR / 'wltc_distribution_shift.csv')

Sauvegarde : ..\results\tables\wltc_distribution_shift.csv


In [11]:
%run "./01_EMS_preprocessing.ipynb"

ROOT_DIR  : C:\Users\Admin\Desktop\Projet_Artemis2
DATA_FILE : C:\Users\Admin\Desktop\Projet_Artemis2\data\Artemis.csv | existe: True
RANDOM_SEED: 42 | DEVICE: cuda
CONFIGURATION DU PROJET HESS

BATTERIE ÉNERGIE
Énergie          : 13709.89 Wh
Tension          : 450.00 V
Capacité         : 30.4664 Ah
Masse            : 55.12 kg
Courant recharge : -14.00 A
Courant décharge : 28.00 A
Configuration    : 125S7P

BATTERIE PUISSANCE
Énergie          : 2987.12 Wh
Tension          : 402.60 V
Capacité         : 7.4196 Ah
Masse            : 50.02 kg
Courant recharge : -130.00 A
Courant décharge : 400.00 A
Configuration    : 122S2P

HESS
Énergie totale   : 16697.01 Wh
Masse totale     : 105.14 kg
Tension du bus   : 402.60 V
Puissance min    : -58638.00 W
Puissance max    : 173640.00 W

MODÈLES
LSTM seul        : 7 → 64 → 3
LSTM NS          : 15 → 64 → 3
GNN simple       : 12 → 32 → 1
MLP simple       : 5 → 64 → 32 → 1
MLP NS           : 17 → 64 → 32 → 1

Device           : cuda
Fichier          : 